# ML_G1G2_P02toP07_DEEP_REENTRY_answer

## Title / 사용법

이 노트북은 `ML_G1G2_P02toP07_DEEP_REENTRY.ipynb`의 정답본이다. 먼저 문제지를 풀고 난 뒤 열어야 한다.

- 정답 포함: 예
- 실행 가능성: fresh kernel 기준 NumPy/Pandas 최소 코드 포함
- 범위: Gate 1 후반 + Gate 2 전체, Gate 3는 Keras 연결 문항에서만 제한적으로 사용


## Source Map

| node_id | 강의 | 이 문제 세트에서 쓰는 역할 |
|---|---:|---|
| `n_ML8.perceptron` | 8강 | Perceptron = weighted sum + activation, `z = w^T x + b` |
| `n_ML8.forward_loss_backward_update` | 8강 | Forward / Loss / Backward / Update 학습 루프 |
| `n_ML8.xor` | 8강 | XOR는 단일 Perceptron으로 선형 분리 불가능 |
| `n_ML8.mlp_xor` | 8강 | hidden layer와 nonlinear activation으로 XOR 해결 |
| `n_ML10.chain_rule` | 10강 | 합성함수의 local derivative 곱 구조 |
| `n_ML10.gradient_descent` | 10강 | gradient는 loss 증가 방향, GD는 반대 방향 |
| `n_ML10.backprop` | 10강 | 계산 그래프 전체에 chain rule 반복 적용 |
| `n_ML11.section_02` | 11강 | 2-2-1 MLP 구조와 파라미터 shape |
| `n_ML11.forward_pass` | 11강 | `z1, a1, z2, a2` forward cache |
| `n_ML11.section_05` | 11강 | output delta와 출력층 gradient |
| `n_ML11.section_06` | 11강 | hidden delta, `W2.T`, `@`와 `*` 구분 |
| `n_ML11.forward_vs_backward` | 11강 | forward 값과 backward gradient의 역할 구분 |
| `n_ML12.backprop` | 12강 | 2-2-1 backprop 패턴을 일반 layer로 확장 |
| `n_ML12.layer_abstraction_activation` | 12강 | `Layer`, `Dense`, `ReLU`, `Sigmoid`, `SoftmaxCE` |
| `n_ML12.vectorized_backprop_batch_dimension_matrix` | 12강 | batch dimension과 `dW = dZ.T @ X` |
| `n_ML12.network_mini_batch` | 12강 | `Network.forward`, reversed `backward`, mini-batch `fit` |
| `n_ML13.keras` | 13강 | 직접 구현 `Network`와 Keras `Sequential/compile/fit` 대응 |


## 정답본 데이터 로드 및 실행 검증 셀

아래 셀은 정답본의 모든 코드 셀이 사용할 데이터와 고정값을 로드한다.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_DIR = Path("data/ML_G1G2_P02toP07_DEEP_REENTRY")
assert DATA_DIR.exists()

xor = pd.read_csv(DATA_DIR / "xor_data.csv")
binary = pd.read_csv(DATA_DIR / "binary_classification_toy.csv")
iris_like = pd.read_csv(DATA_DIR / "iris_like_3class_toy.csv")
params = np.load(DATA_DIR / "mlp_221_fixed.npz")
batch = np.load(DATA_DIR / "minibatch_shape_examples.npz")
metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))

assert xor.shape == (4, 6)
assert binary.shape[1] == 4
assert iris_like["class_id"].nunique() == 3
assert set(params.files) == {"W1", "b1", "W2", "b2", "x", "y"}
assert set(batch.files) == {"X", "W", "b", "dZ"}

print("data assets OK")
print("xor", xor.shape, "binary", binary.shape, "iris_like", iris_like.shape)


## 문항별 정답과 해설

각 문항은 다음 10개 구조를 따른다.

1. 정답
2. 근거
3. 수식 전개
4. shape 설명
5. 자주 하는 오답
6. 왜 그런 오답이 나오는가
7. 정답형 사고법
8. Gate 연결 노드
9. 재시도 미니문제
10. 오답튜터 피드백 문장


In [ ]:
def sigmoid(t):
    return 1 / (1 + np.exp(-t))

x = params["x"]
y = params["y"]
W1 = params["W1"]
b1 = params["b1"]
W2 = params["W2"]
b2 = params["b2"]

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = z2.copy()
loss = 0.5 * np.sum((y - a2) ** 2)
delta2 = a2 - y
dW2 = np.outer(delta2, a1)
db2 = delta2.copy()
dL_da1 = W2.T @ delta2
delta1 = dL_da1 * a1 * (1 - a1)
dW1 = np.outer(delta1, x)
db1 = delta1.copy()

def show(name, value):
    print(f"{name}: {np.asarray(value).round(6)} shape={np.asarray(value).shape}")

for name, value in [
    ("z1", z1), ("a1", a1), ("z2", z2), ("a2", a2), ("loss", np.array(loss)),
    ("delta2", delta2), ("dW2", dW2), ("db2", db2),
    ("dL_da1", dL_da1), ("delta1", delta1), ("dW1", dW1), ("db1", db1),
]:
    show(name, value)

assert np.allclose(z1, [0.3, 0.7])
assert np.allclose(a1, [0.57444252, 0.66818777])
assert np.allclose(a2, [0.68813492])
assert np.isclose(loss, 0.23676484)
assert dW1.shape == W1.shape and db1.shape == b1.shape
assert dW2.shape == W2.shape and db2.shape == b2.shape


### 문제 1. Q01-A

#### 1. 정답
`z = w^T x + b`, `a = phi(z)`. Perceptron은 weighted sum 뒤에 activation을 붙인 인공 뉴런 1개다. MLP는 이런 뉴런을 layer 단위로 여러 개 연결하고, hidden layer와 nonlinear activation으로 입력 표현을 바꾼다.

#### 2. 근거
8강은 Perceptron을 weighted sum + activation으로 두고, XOR 한계를 hidden layer/MLP 필요성으로 연결한다.

#### 3. 수식 전개
단일 뉴런: `z = sum_j w_j x_j + b`, `a = phi(z)`. MLP: `a^(l)=phi(W^(l)a^(l-1)+b^(l))`.

#### 4. shape 설명
`x: (D,)`, `w: (D,)`, `b: scalar`, `z/a: scalar`. 여러 뉴런이면 `W: (H,D)`, `b: (H,)`, `a: (H,)`.

#### 5. 자주 하는 오답
activation을 threshold로만 설명하거나, MLP를 단순히 Perceptron을 많이 더한 것으로만 설명한다.

#### 6. 왜 그런 오답이 나오는가
weighted sum과 nonlinear transformation의 역할을 분리하지 못해서 생기는 오답이다.

#### 7. 정답형 사고법
먼저 선형 가중합을 쓰고, 그 다음 activation이 비선형 표현을 만드는 이유를 붙인다.

#### 8. Gate 연결 노드
G1-P02

#### 9. 재시도 미니문제
`x`가 3차원이고 hidden neuron이 4개라면 `W`와 `b` shape를 쓰시오.

#### 10. 오답튜터 피드백 문장
계산식에는 `w, x, b`가, 구조 설명에는 hidden layer와 nonlinear activation이 반드시 들어가야 합니다.


### 문제 2. Q02-B

#### 1. 정답
AND/OR/NAND는 class를 직선 하나로 나눌 수 있지만 XOR는 양성 class가 대각선에 있어 직선 하나로 분리할 수 없다. hidden layer는 원래 입력을 새 좌표 또는 feature 표현으로 변환해, 변환된 공간에서 분리가 가능하게 만든다.

#### 2. 근거
8강 XOR 노드는 단일 Perceptron의 선형 분리 한계와 MLP의 XOR 해결을 연결한다.

#### 3. 수식 전개
XOR는 `x1 != x2`일 때 1이다. 논리 조합으로는 `XOR = AND(OR(x1,x2), NAND(x1,x2))`처럼 hidden 논리 feature를 만들 수 있다.

#### 4. shape 설명
XOR 입력 `X: (4,2)`, target `y: (4,)` 또는 `(4,1)`.

#### 5. 자주 하는 오답
“XOR가 비선형이라서”만 쓰고 왜 선형 분리 불가능한지 설명하지 않는다.

#### 6. 왜 그런 오답이 나오는가
데이터 배치와 decision boundary를 연결하지 못했기 때문이다.

#### 7. 정답형 사고법
진리표를 좌표평면에 놓고, 직선 하나가 같은 class를 동시에 묶을 수 있는지 먼저 본다.

#### 8. Gate 연결 노드
G1-P02

#### 9. 재시도 미니문제
NAND와 OR 출력을 hidden feature로 만들면 XOR output을 어떤 논리로 얻는지 쓰시오.

#### 10. 오답튜터 피드백 문장
XOR 설명에는 “대각선 배치”와 “hidden representation 변환”이 함께 있어야 합니다.


### 문제 3. Q03-C

#### 1. 정답
1은 O. 2는 X: gradient는 loss 증가 방향이다. 3은 X: gradient descent는 loss를 줄이기 위해 gradient 반대 방향으로 이동한다. 4는 O: optimizer는 계산된 gradient를 사용해 parameter를 update하는 규칙이다.

#### 2. 근거
10강 gradient descent와 13강 optimizer 책임 분리.

#### 3. 수식 전개
`theta_new = theta - lr * grad`. 여기서 `grad = dL/dtheta`는 증가 방향이므로 빼야 한다.

#### 4. shape 설명
`theta`와 `grad`는 같은 shape다. 예: `W: (H,D)`, `dW: (H,D)`.

#### 5. 자주 하는 오답
optimizer가 gradient를 계산한다고 쓰거나, gradient를 감소 방향으로 착각한다.

#### 6. 왜 그런 오답이 나오는가
backprop과 optimizer step을 한 덩어리로 암기했기 때문이다.

#### 7. 정답형 사고법
backward는 gradient를 계산하고, optimizer/update는 그 gradient를 사용해 parameter를 움직인다.

#### 8. Gate 연결 노드
G1-P03, G1-P05, 연결 n_ML13.keras

#### 9. 재시도 미니문제
`W <- W + lr*dW`가 왜 보통 틀린 update인지 설명하시오.

#### 10. 오답튜터 피드백 문장
gradient 방향과 update 방향을 반드시 반대로 구분하세요.


### 문제 4. Q04-D

#### 1. 정답
`u=g(x)`, `y=f(u)`이면 `dy/dx = dy/du * du/dx = f'(g(x))g'(x)`이다. backprop에서는 “위에서 온 gradient × 자기 local derivative”를 계산한다. 경로가 여러 개이면 각 경로의 곱 기여를 합한다.

#### 2. 근거
10강 chain rule과 계산 그래프 backprop.

#### 3. 수식 전개
single path: upstream gradient times local derivative. multiple paths: sum of path contributions.

#### 4. shape 설명
local derivative가 scalar면 원소별 곱, dense mapping이면 matrix multiply로 upstream gradient를 변환한다.

#### 5. 자주 하는 오답
local derivative만 쓰고 upstream gradient를 빠뜨린다.

#### 6. 왜 그런 오답이 나오는가
미분 공식을 함수 하나의 기울기로만 보고 계산 그래프의 연결을 보지 못해서다.

#### 7. 정답형 사고법
현재 노드는 전체 loss를 직접 보지 않고, 위에서 온 민감도와 자기 local 변화율만 결합한다.

#### 8. Gate 연결 노드
G1-P04

#### 9. 재시도 미니문제
`L -> a -> z -> W` 경로에서 각 local derivative 이름을 쓰시오.

#### 10. 오답튜터 피드백 문장
backprop 문장에는 upstream gradient와 local derivative가 모두 필요합니다.


### 문제 5. Q05-E

#### 1. 정답
순서는 Forward → Loss → Backward → Update. Forward는 예측과 cache를 만들고, Loss는 scalar loss와 시작 gradient의 근거를 만들며, Backward는 `dW/db` 같은 gradient를 만들고, Update는 parameter를 이동한다.

#### 2. 근거
8강 학습 루프와 11강 forward/backward 총정리.

#### 3. 수식 전개
`y_hat = f(X;theta)`, `L = loss(y,y_hat)`, `grads = backward(cache, y)`, `theta <- theta - lr*grads`.

#### 4. shape 설명
cache: `z1 (2,)`, `a1 (2,)`, `z2/a2 (1,)`; grads는 각 parameter와 같은 shape다.

#### 5. 자주 하는 오답
cache와 gradient를 같은 것으로 쓰거나, update에서 gradient를 새로 계산한다고 쓴다.

#### 6. 왜 그런 오답이 나오는가
학습 루프의 책임 분리가 흐려졌기 때문이다.

#### 7. 정답형 사고법
Forward는 값 저장, Backward는 책임 계산, Update는 parameter 이동으로 나눈다.

#### 8. Gate 연결 노드
G1-P05

#### 9. 재시도 미니문제
`a1`을 cache하지 않으면 출력층 `dW2`와 sigmoid derivative 계산에서 무엇이 불편한지 쓰시오.

#### 10. 오답튜터 피드백 문장
cache는 backward 계산을 위해 forward 중간값을 보관하는 장치입니다.


### 문제 6. Q06-F0

#### 1. 정답
`x (2,)`, `W1 (2,2)`, `b1 (2,)`, `z1/a1 (2,)`, `W2 (1,2)`, `b2 (1,)`, `z2/a2 (1,)`, `loss scalar`.

#### 2. 근거
11강 2-2-1 네트워크 구조와 표기.

#### 3. 수식 전개
`z1 = W1 @ x + b1`, `a1 = sigmoid(z1)`, `z2 = W2 @ a1 + b2`, `a2 = z2`.

#### 4. shape 설명
`(2,2)@(2,) -> (2,)`, `(1,2)@(2,) -> (1,)`.

#### 5. 자주 하는 오답
`W1`을 `(input, hidden)`으로 두고 `x @ W1` convention과 섞는다.

#### 6. 왜 그런 오답이 나오는가
단일 샘플 convention과 batch convention을 동시에 섞었기 때문이다.

#### 7. 정답형 사고법
현재 문제에서는 행이 출력 뉴런, 열이 입력 값이다.

#### 8. Gate 연결 노드
G2-P01

#### 9. 재시도 미니문제
hidden neuron이 3개라면 `W1`, `b1`, `W2` shape가 어떻게 바뀌는가?

#### 10. 오답튜터 피드백 문장
먼저 convention을 고정한 뒤 모든 shape를 거기에 맞춰야 합니다.


### 문제 7. Q07-F

#### 1. 정답
`z1=[0.3, 0.7]`, `a1=[0.5744, 0.6682]`, `z2/a2=[0.6881]`, `loss=0.2368`.

#### 2. 근거
11강 forward pass의 고정 숫자 예제.

#### 3. 수식 전개
`z1 = W1 @ x + b1`, `a1 = sigmoid(z1)`, `z2 = W2 @ a1 + b2`, `loss = 0.5*(y-a2)^2`.

#### 4. shape 설명
`z1/a1 (2,)`, `z2/a2 (1,)`, `loss scalar`.

#### 5. 자주 하는 오답
출력층에도 sigmoid를 적용하거나 loss에서 1/2를 빼먹는다.

#### 6. 왜 그런 오답이 나오는가
문제의 output layer가 linear라는 조건을 놓쳤기 때문이다.

#### 7. 정답형 사고법
forward는 오른쪽으로 값만 계산하고, 아직 gradient는 계산하지 않는다.

#### 8. Gate 연결 노드
G2-P01

#### 9. 재시도 미니문제
`y=1`이라면 같은 `a2`에서 loss만 다시 계산하시오.

#### 10. 오답튜터 피드백 문장
forward cache 값은 이후 backward에서 그대로 재사용됩니다.


### 문제 8. Q08-Bridge

#### 1. 정답
출력층 경로는 `W2 -> z2 -> a2 -> L`이라 짧고, 은닉층 경로는 `W1 -> z1 -> a1 -> z2 -> a2 -> L`이라 sigmoid local derivative와 `W2`를 추가로 거친다.

#### 2. 근거
10강 chain rule과 11강 짧은 체인/긴 체인 대비.

#### 3. 수식 전개
출력층: `dL/dW2_i = delta2 * a1_i`. 은닉층: `dL/dW1_ij = delta1_i * x_j`, `delta1 = (W2.T @ delta2) * sigmoid'(z1)`.

#### 4. shape 설명
출력층 `dW2 (1,2)`, 은닉층 `dW1 (2,2)`.

#### 5. 자주 하는 오답
은닉층 gradient를 출력층처럼 `delta2*x`로 계산한다.

#### 6. 왜 그런 오답이 나오는가
은닉층이 loss에 직접 닿지 않고 출력층을 통해 간접 연결된다는 점을 놓친다.

#### 7. 정답형 사고법
구하려는 parameter에서 loss까지 실제 경로를 먼저 그린다.

#### 8. Gate 연결 노드
G1-P04, G2-P02~P03

#### 9. 재시도 미니문제
`W1[1,0]`에서 loss까지 가는 경로를 쓰시오.

#### 10. 오답튜터 피드백 문장
은닉층에서는 반드시 출력층 weight와 activation local derivative가 끼어듭니다.


### 문제 9. Q09-G

#### 1. 정답
`delta2 = a2 - y = 0.6881`. `dW2=[[0.3953, 0.4598]]`, `db2=[0.6881]`.

#### 2. 근거
11강 출력층 짧은 체인.

#### 3. 수식 전개
`d/da2 0.5(y-a2)^2 = a2-y`; `dW2 = outer(delta2, a1)`, `db2 = delta2`.

#### 4. shape 설명
`delta2 (1,)`, `dW2 (1,2)`, `db2 (1,)`.

#### 5. 자주 하는 오답
`delta2 = y-a2`로 부호를 반대로 쓴다.

#### 6. 왜 그런 오답이 나오는가
내부 미분 `d(y-a2)/da2 = -1`을 놓쳐서 생긴다.

#### 7. 정답형 사고법
loss 미분 부호를 먼저 확정하고, 그 다음 “delta × 입력” 패턴을 적용한다.

#### 8. Gate 연결 노드
G2-P02

#### 9. 재시도 미니문제
`a2=0.2, y=1`이면 `delta2`의 부호는 무엇인가?

#### 10. 오답튜터 피드백 문장
출력층 delta의 부호는 update 방향 전체를 바꾸므로 반드시 확인해야 합니다.


### 문제 10. Q10-H

#### 1. 정답
`dL/da1=[0.3441, 0.4129]`, `delta1=[0.0841, 0.0915]`. `W2.T`는 출력 gradient `(1,)`을 hidden activation 방향 `(2,)`으로 되돌리기 위해 필요하다. `@`는 행렬곱, `*`는 elementwise 곱이다.

#### 2. 근거
11강 은닉층 긴 체인과 `W2.T`, `@` vs `*`.

#### 3. 수식 전개
`dL_da1 = W2.T @ delta2`; `delta1 = dL_da1 * a1 * (1-a1)`.

#### 4. shape 설명
`W2.T (2,1) @ delta2 (1,) -> (2,)`; sigmoid derivative도 `(2,)`라 elementwise 곱한다.

#### 5. 자주 하는 오답
`W2 @ delta2`를 쓰거나 sigmoid derivative 위치에 `@`를 쓴다.

#### 6. 왜 그런 오답이 나오는가
forward 방향과 backward 방향이 반대라는 점과 elementwise derivative를 구분하지 못해서다.

#### 7. 정답형 사고법
선형 layer를 거슬러 올라갈 때는 transpose, activation local derivative는 같은 위치끼리 곱한다.

#### 8. Gate 연결 노드
G2-P03

#### 9. 재시도 미니문제
`W2`가 `(1,3)`이면 `W2.T @ delta2` 결과 shape는 무엇인가?

#### 10. 오답튜터 피드백 문장
은닉 delta의 핵심은 `W2.T`와 sigmoid local derivative를 분리해서 쓰는 것입니다.


### 문제 11. Q11-I

#### 1. 정답
`dW1=[[0.0841, 0.0841],
 [0.0915, 0.0915]]`, `db1=[0.0841, 0.0915]`. `x=[1,1]`이라 각 row의 두 항이 같지만 shape는 `(2,2)`이다.

#### 2. 근거
11강 은닉층 dW/db 계산.

#### 3. 수식 전개
`dW1 = outer(delta1, x)`, `db1 = delta1`.

#### 4. shape 설명
`outer((2,), (2,)) -> (2,2)`, `db1 (2,)`.

#### 5. 자주 하는 오답
`dW1`을 `delta1 * x`로 써서 `(2,)` 벡터처럼 처리한다.

#### 6. 왜 그런 오답이 나오는가
단일 예제에서 x 값이 모두 1이라 행렬 구조가 눈에 덜 보인다.

#### 7. 정답형 사고법
각 hidden neuron delta가 각 input feature와 쌍을 이루므로 모든 쌍의 곱, 즉 outer product다.

#### 8. Gate 연결 노드
G2-P04 전 단계

#### 9. 재시도 미니문제
`x=[2,-1]`이면 `dW1`의 각 row가 어떻게 달라지는지 식으로 쓰시오.

#### 10. 오답튜터 피드백 문장
수치가 우연히 같아져도 gradient tensor의 shape는 보존해야 합니다.


### 문제 12. Q12-BackpropSummary

#### 1. 정답
`grads`는 `dW1 (2,2)`, `db1 (2,)`, `dW2 (1,2)`, `db2 (1,)`이다. update는 `param <- param - lr * grad`. gradient check는 수치미분과 backward의 해석적 gradient가 일치하는지 비교한다.

#### 2. 근거
11강 forward vs backward 총정리와 gradient check.

#### 3. 수식 전개
`numeric_grad ≈ (L(w+eps)-L(w-eps))/(2eps)`를 `backward`의 gradient와 비교한다.

#### 4. shape 설명
모든 parameter gradient는 parameter와 같은 shape다.

#### 5. 자주 하는 오답
gradient check를 validation accuracy나 loss 감소 확인으로 설명한다.

#### 6. 왜 그런 오답이 나오는가
구현 검증과 모델 성능 평가를 혼동했기 때문이다.

#### 7. 정답형 사고법
gradient check는 “내가 유도하고 구현한 미분식이 맞는가”를 보는 디버깅 장치다.

#### 8. Gate 연결 노드
G2-P02~P04

#### 9. 재시도 미니문제
gradient check가 실패했을 때 `W2.T`, 부호, `*`/`@` 중 무엇을 먼저 볼지 순서를 정하시오.

#### 10. 오답튜터 피드백 문장
gradient check는 학습 결과가 아니라 backward 구현의 신뢰도를 검증합니다.


### 문제 13. Q13-JK

#### 1. 정답
`self.dW = dZ.T @ X`, `self.db = dZ.sum(axis=0)`, `dX = dZ @ W`. ReLU와 Sigmoid는 parameter가 없고 update 대상이 아니다.

#### 2. 근거
12강 Dense Layer와 Activation Layer.

#### 3. 수식 전개
`Z = X @ W.T + b`; backward: `dW=dZ.T@X`, `db=sum_B dZ`, `dX=dZ@W`. ReLU: `dout*(x>0)`, Sigmoid: `dout*out*(1-out)`.

#### 4. shape 설명
`dW (H,D)`, `db (H,)`, `dX (B,D)`. ReLU/Sigmoid 입출력 gradient shape는 입력과 같다.

#### 5. 자주 하는 오답
`dW = X.T @ dZ`를 쓰고 shape `(D,H)`로 만든다.

#### 6. 왜 그런 오답이 나오는가
W convention을 `(D,H)`로 둔 다른 교재 식과 섞었기 때문이다.

#### 7. 정답형 사고법
항상 결과 `dW`가 현재 `W`와 같은 shape인지 확인한다.

#### 8. Gate 연결 노드
G2-P04, G2-P05

#### 9. 재시도 미니문제
`B=5,D=4,H=3`일 때 `dZ.T @ X` shape를 쓰시오.

#### 10. 오답튜터 피드백 문장
Dense backward 공식은 convention에 따라 달라 보이므로 shape 검산이 필수입니다.


In [ ]:
X = batch["X"]
W = batch["W"]
b = batch["b"]
dZ = batch["dZ"]
Z = X @ W.T + b
dW = dZ.T @ X
db = dZ.sum(axis=0)
dX = dZ @ W

print("X", X.shape, "W", W.shape, "b", b.shape, "Z", Z.shape)
print("dZ", dZ.shape, "dW", dW.shape, "db", db.shape, "dX", dX.shape)
assert dW.shape == W.shape
assert db.shape == b.shape
assert dX.shape == X.shape

relu_x = np.array([[-1.0, 0.0, 2.0]])
relu_grad = np.ones_like(relu_x)
relu_back = relu_grad * (relu_x > 0)
sig_out = sigmoid(relu_x)
sig_back = relu_grad * sig_out * (1 - sig_out)
print("relu_back", relu_back)
print("sigmoid_back", sig_back.round(4))


### 문제 14. Q14-L

#### 1. 정답
SoftmaxCE는 logits를 class probability로 바꾸고 cross entropy loss를 계산한다. one-hot target 기준 backward는 batch 평균이면 `(p-y)/B`, 평균이 아니면 `p-y`다. binary sigmoid는 출력 1개 확률, multiclass softmax는 K개 확률분포다.

#### 2. 근거
12강 SoftmaxCE와 13강 분류 loss 연결.

#### 3. 수식 전개
`probs = exp(logits - max)/sum(exp(...))`; `CE = -mean(sum(y*log(p)))`; `dlogits=(probs-y)/B`.

#### 4. shape 설명
`logits/probs/y_onehot: (B,K)`, `loss scalar`, `dlogits (B,K)`.

#### 5. 자주 하는 오답
softmax를 class별 독립 sigmoid로 생각하거나 `/B` 위치를 loss 평균과 분리하지 못한다.

#### 6. 왜 그런 오답이 나오는가
확률분포의 class 축 합 1 조건과 batch 평균 convention을 놓쳤기 때문이다.

#### 7. 정답형 사고법
multiclass는 K개 score를 서로 비교해 하나의 분포를 만든다.

#### 8. Gate 연결 노드
G2-P05

#### 9. 재시도 미니문제
`B=4,K=3`이면 logits와 one-hot target shape를 쓰시오.

#### 10. 오답튜터 피드백 문장
SoftmaxCE backward의 간단한 형태는 softmax와 CE를 결합했을 때 나옵니다.


In [ ]:
def softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

logits = np.array([[2.0, 0.5, -1.0], [0.1, 1.2, 0.3]])
y_onehot = np.array([[1, 0, 0], [0, 1, 0]], dtype=float)
probs = softmax(logits)
loss_ce = -np.sum(y_onehot * np.log(probs + 1e-12)) / logits.shape[0]
dlogits = (probs - y_onehot) / logits.shape[0]

print("probs", probs.round(4))
print("row sums", probs.sum(axis=1))
print("loss", round(float(loss_ce), 6))
print("dlogits", dlogits.round(4), dlogits.shape)
assert np.allclose(probs.sum(axis=1), 1.0)
assert dlogits.shape == logits.shape


### 문제 15. Q15-MN

#### 1. 정답
단일 샘플의 `outer(delta,x)`를 batch별로 모두 더하면 `dZ.T @ X`가 된다. `Network.forward`는 앞에서 뒤로, `Network.backward`는 뒤에서 앞으로 돈다. 각 layer는 위층 gradient와 자기 local derivative만 알면 된다.

#### 2. 근거
12강 vectorized backprop과 network mini-batch.

#### 3. 수식 전개
`Z=X@W.T+b`, `dW=dZ.T@X`, `db=dZ.sum(axis=0)`, `dX=dZ@W`.

#### 4. shape 설명
`X (B,D)`, `W (H,D)`, `Z/dZ (B,H)`, `dW (H,D)`, `db (H,)`, `dX (B,D)`.

#### 5. 자주 하는 오답
batch 축을 feature 축으로 보고 `axis=1`로 bias gradient를 더한다.

#### 6. 왜 그런 오답이 나오는가
batch dimension 0번째 축이라는 convention을 고정하지 않았기 때문이다.

#### 7. 정답형 사고법
batch는 샘플 여러 개를 한 번에 쌓은 것이며, 각 샘플 gradient를 행렬곱으로 합친다.

#### 8. Gate 연결 노드
G2-P06, G2-P07

#### 9. 재시도 미니문제
`B=8,D=2,H=4`에서 `X`, `W`, `Z`, `dW` shape를 쓰시오.

#### 10. 오답튜터 피드백 문장
Network abstraction은 chain rule을 layer list 순회 구조로 고정한 것입니다.


In [ ]:
class Dense:
    def __init__(self, n_in, n_out, seed=0):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0, 0.1, size=(n_out, n_in))
        self.b = np.zeros(n_out)
    def forward(self, X):
        self.X = X
        return X @ self.W.T + self.b
    def backward(self, dZ):
        self.dW = dZ.T @ self.X
        self.db = dZ.sum(axis=0)
        return dZ @ self.W
    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db

class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)
    def backward(self, dY):
        return dY * (self.X > 0)
    def update(self, lr):
        pass

class Network:
    def __init__(self):
        self.layers = []
    def add(self, layer):
        self.layers.append(layer)
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad
    def update(self, lr):
        for layer in self.layers:
            layer.update(lr)

net = Network()
net.add(Dense(2, 4, seed=1))
net.add(ReLU())
net.add(Dense(4, 1, seed=2))
X_small = binary[["feature_1", "feature_2"]].to_numpy()[:8]
out = net.forward(X_small)
upstream = np.ones_like(out) / out.shape[0]
dX_first = net.backward(upstream)
net.update(0.01)
print("out", out.shape, "dX_first", dX_first.shape)
assert out.shape == (8, 1)
assert dX_first.shape == X_small.shape


### 문제 16. Q16-OP

#### 1. 정답
`X = [feature_1, feature_2]`, `y = target`. input shape는 `(None,2)` 또는 직접 구현 batch에서 `(B,2)`. binary output은 `Dense(1)` + sigmoid/BCE 또는 2-logit softmax/CE가 가능하다. metric은 accuracy. validation은 학습에 쓰지 않은 데이터에서 일반화와 overfitting을 확인하기 위해 필요하다. `Network()`는 `keras.Sequential()`, `net.add(Dense(...))`는 `model.add(Dense(...))`, `ReLU()`는 `activation="relu"`, 직접 train loop는 `fit`, 직접 평가는 `evaluate`에 대응한다.

#### 2. 근거
13강 Keras 모델링의 공통 절차와 12강 Network 구조.

#### 3. 수식 전개
직접 구현 흐름: `forward -> loss.forward -> loss.backward -> net.backward -> optimizer/update`. Keras 흐름: `model.compile(...) -> model.fit(...) -> model.evaluate(...)`.

#### 4. shape 설명
`B=8`이면 `X_batch (8,2)`, binary sigmoid target은 `(8,1)` 또는 `(8,)` convention 중 하나를 명시한다.

#### 5. 자주 하는 오답
compile이 학습 실행이라고 쓰거나, optimizer가 gradient를 계산한다고 쓴다.

#### 6. 왜 그런 오답이 나오는가
Keras API 단계를 직접 구현 학습 루프의 책임과 대응시키지 못해서다.

#### 7. 정답형 사고법
먼저 문제 유형으로 output/loss/metric을 정하고, 그 다음 API 이름을 직접 구현 책임에 연결한다.

#### 8. Gate 연결 노드
G2-P07, 연결 n_ML13.keras

#### 9. 재시도 미니문제
3-class toy data라면 output layer, loss, y 표현이 어떻게 바뀌는지 쓰시오.

#### 10. 오답튜터 피드백 문장
Keras 연결 문항의 핵심은 문법 암기가 아니라 직접 구현 구조와 책임 단위의 대응입니다.


In [ ]:
X_cols = ["feature_1", "feature_2"]
y_col = "target"
X_all = binary[X_cols].to_numpy(dtype=float)
y_all = binary[y_col].to_numpy(dtype=float).reshape(-1, 1)
train_mask = binary["split"].eq("train").to_numpy()
X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[~train_mask], y_all[~train_mask]

print("X_train", X_train.shape, "y_train", y_train.shape)
print("X_val", X_val.shape, "y_val", y_val.shape)
print("Keras mapping: Network -> Sequential, add(Dense) -> model.add(Dense), train loop -> fit, eval -> evaluate")
assert X_train.shape[1] == 2
assert y_train.shape[1] == 1


## 오답튜터 표

아래 표는 채점 후 어떤 오답을 어떤 힌트로 되돌릴지 정리한 기준표다.


In [ ]:
feedback_rows = [{'question_id': 'Q01-A', 'assessed_concept': 'G1-P02', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'activation을 threshold로만 설명하거나, MLP를 단순히 Perceptron을 많이 더한 것으로만 설명한다.', 'likely_misconception': 'weighted sum과 nonlinear transformation의 역할을 분리하지 못해서 생기는 오답이다.', 'correction_hint_only': '계산식에는 `w, x, b`가, 구조 설명에는 hidden layer와 nonlinear activation이 반드시 들어가야 합니다.', 'retry_prompt': '`x`가 3차원이고 hidden neuron이 4개라면 `W`와 `b` shape를 쓰시오.', 'gate_node': 'G1-P02', 'source_lecture': '8강'}, {'question_id': 'Q02-B', 'assessed_concept': 'G1-P02', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '“XOR가 비선형이라서”만 쓰고 왜 선형 분리 불가능한지 설명하지 않는다.', 'likely_misconception': '데이터 배치와 decision boundary를 연결하지 못했기 때문이다.', 'correction_hint_only': 'XOR 설명에는 “대각선 배치”와 “hidden representation 변환”이 함께 있어야 합니다.', 'retry_prompt': 'NAND와 OR 출력을 hidden feature로 만들면 XOR output을 어떤 논리로 얻는지 쓰시오.', 'gate_node': 'G1-P02', 'source_lecture': '8강'}, {'question_id': 'Q03-C', 'assessed_concept': 'G1-P03, G1-P05, 연결 n_ML13.keras', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'optimizer가 gradient를 계산한다고 쓰거나, gradient를 감소 방향으로 착각한다.', 'likely_misconception': 'backprop과 optimizer step을 한 덩어리로 암기했기 때문이다.', 'correction_hint_only': 'gradient 방향과 update 방향을 반드시 반대로 구분하세요.', 'retry_prompt': '`W <- W + lr*dW`가 왜 보통 틀린 update인지 설명하시오.', 'gate_node': 'G1-P03, G1-P05, 연결 n_ML13.keras', 'source_lecture': '10강'}, {'question_id': 'Q04-D', 'assessed_concept': 'G1-P04', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'local derivative만 쓰고 upstream gradient를 빠뜨린다.', 'likely_misconception': '미분 공식을 함수 하나의 기울기로만 보고 계산 그래프의 연결을 보지 못해서다.', 'correction_hint_only': 'backprop 문장에는 upstream gradient와 local derivative가 모두 필요합니다.', 'retry_prompt': '`L -> a -> z -> W` 경로에서 각 local derivative 이름을 쓰시오.', 'gate_node': 'G1-P04', 'source_lecture': '10강'}, {'question_id': 'Q05-E', 'assessed_concept': 'G1-P05', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'cache와 gradient를 같은 것으로 쓰거나, update에서 gradient를 새로 계산한다고 쓴다.', 'likely_misconception': '학습 루프의 책임 분리가 흐려졌기 때문이다.', 'correction_hint_only': 'cache는 backward 계산을 위해 forward 중간값을 보관하는 장치입니다.', 'retry_prompt': '`a1`을 cache하지 않으면 출력층 `dW2`와 sigmoid derivative 계산에서 무엇이 불편한지 쓰시오.', 'gate_node': 'G1-P05', 'source_lecture': '11강'}, {'question_id': 'Q06-F0', 'assessed_concept': 'G2-P01', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '`W1`을 `(input, hidden)`으로 두고 `x @ W1` convention과 섞는다.', 'likely_misconception': '단일 샘플 convention과 batch convention을 동시에 섞었기 때문이다.', 'correction_hint_only': '먼저 convention을 고정한 뒤 모든 shape를 거기에 맞춰야 합니다.', 'retry_prompt': 'hidden neuron이 3개라면 `W1`, `b1`, `W2` shape가 어떻게 바뀌는가?', 'gate_node': 'G2-P01', 'source_lecture': '11강'}, {'question_id': 'Q07-F', 'assessed_concept': 'G2-P01', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '출력층에도 sigmoid를 적용하거나 loss에서 1/2를 빼먹는다.', 'likely_misconception': '문제의 output layer가 linear라는 조건을 놓쳤기 때문이다.', 'correction_hint_only': 'forward cache 값은 이후 backward에서 그대로 재사용됩니다.', 'retry_prompt': '`y=1`이라면 같은 `a2`에서 loss만 다시 계산하시오.', 'gate_node': 'G2-P01', 'source_lecture': '11강'}, {'question_id': 'Q08-Bridge', 'assessed_concept': 'G1-P04, G2-P02~P03', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '은닉층 gradient를 출력층처럼 `delta2*x`로 계산한다.', 'likely_misconception': '은닉층이 loss에 직접 닿지 않고 출력층을 통해 간접 연결된다는 점을 놓친다.', 'correction_hint_only': '은닉층에서는 반드시 출력층 weight와 activation local derivative가 끼어듭니다.', 'retry_prompt': '`W1[1,0]`에서 loss까지 가는 경로를 쓰시오.', 'gate_node': 'G1-P04, G2-P02~P03', 'source_lecture': '11강'}, {'question_id': 'Q09-G', 'assessed_concept': 'G2-P02', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '`delta2 = y-a2`로 부호를 반대로 쓴다.', 'likely_misconception': '내부 미분 `d(y-a2)/da2 = -1`을 놓쳐서 생긴다.', 'correction_hint_only': '출력층 delta의 부호는 update 방향 전체를 바꾸므로 반드시 확인해야 합니다.', 'retry_prompt': '`a2=0.2, y=1`이면 `delta2`의 부호는 무엇인가?', 'gate_node': 'G2-P02', 'source_lecture': '11강'}, {'question_id': 'Q10-H', 'assessed_concept': 'G2-P03', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '`W2 @ delta2`를 쓰거나 sigmoid derivative 위치에 `@`를 쓴다.', 'likely_misconception': 'forward 방향과 backward 방향이 반대라는 점과 elementwise derivative를 구분하지 못해서다.', 'correction_hint_only': '은닉 delta의 핵심은 `W2.T`와 sigmoid local derivative를 분리해서 쓰는 것입니다.', 'retry_prompt': '`W2`가 `(1,3)`이면 `W2.T @ delta2` 결과 shape는 무엇인가?', 'gate_node': 'G2-P03', 'source_lecture': '11강'}, {'question_id': 'Q11-I', 'assessed_concept': 'G2-P04 전 단계', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '`dW1`을 `delta1 * x`로 써서 `(2,)` 벡터처럼 처리한다.', 'likely_misconception': '단일 예제에서 x 값이 모두 1이라 행렬 구조가 눈에 덜 보인다.', 'correction_hint_only': '수치가 우연히 같아져도 gradient tensor의 shape는 보존해야 합니다.', 'retry_prompt': '`x=[2,-1]`이면 `dW1`의 각 row가 어떻게 달라지는지 식으로 쓰시오.', 'gate_node': 'G2-P04 전 단계', 'source_lecture': '11강'}, {'question_id': 'Q12-BackpropSummary', 'assessed_concept': 'G2-P02~P04', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'gradient check를 validation accuracy나 loss 감소 확인으로 설명한다.', 'likely_misconception': '구현 검증과 모델 성능 평가를 혼동했기 때문이다.', 'correction_hint_only': 'gradient check는 학습 결과가 아니라 backward 구현의 신뢰도를 검증합니다.', 'retry_prompt': 'gradient check가 실패했을 때 `W2.T`, 부호, `*`/`@` 중 무엇을 먼저 볼지 순서를 정하시오.', 'gate_node': 'G2-P02~P04', 'source_lecture': '11강'}, {'question_id': 'Q13-JK', 'assessed_concept': 'G2-P04, G2-P05', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': '`dW = X.T @ dZ`를 쓰고 shape `(D,H)`로 만든다.', 'likely_misconception': 'W convention을 `(D,H)`로 둔 다른 교재 식과 섞었기 때문이다.', 'correction_hint_only': 'Dense backward 공식은 convention에 따라 달라 보이므로 shape 검산이 필수입니다.', 'retry_prompt': '`B=5,D=4,H=3`일 때 `dZ.T @ X` shape를 쓰시오.', 'gate_node': 'G2-P04, G2-P05', 'source_lecture': '12강'}, {'question_id': 'Q14-L', 'assessed_concept': 'G2-P05', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'softmax를 class별 독립 sigmoid로 생각하거나 `/B` 위치를 loss 평균과 분리하지 못한다.', 'likely_misconception': '확률분포의 class 축 합 1 조건과 batch 평균 convention을 놓쳤기 때문이다.', 'correction_hint_only': 'SoftmaxCE backward의 간단한 형태는 softmax와 CE를 결합했을 때 나옵니다.', 'retry_prompt': '`B=4,K=3`이면 logits와 one-hot target shape를 쓰시오.', 'gate_node': 'G2-P05', 'source_lecture': '12강'}, {'question_id': 'Q15-MN', 'assessed_concept': 'G2-P06, G2-P07', 'expected_answer_type': '개념+수식+shape 설명', 'common_wrong_answer': 'batch 축을 feature 축으로 보고 `axis=1`로 bias gradient를 더한다.', 'likely_misconception': 'batch dimension 0번째 축이라는 convention을 고정하지 않았기 때문이다.', 'correction_hint_only': 'Network abstraction은 chain rule을 layer list 순회 구조로 고정한 것입니다.', 'retry_prompt': '`B=8,D=2,H=4`에서 `X`, `W`, `Z`, `dW` shape를 쓰시오.', 'gate_node': 'G2-P06, G2-P07', 'source_lecture': '12강'}, {'question_id': 'Q16-OP', 'assessed_concept': 'G2-P07, 연결 n_ML13.keras', 'expected_answer_type': '종합 설계 판단', 'common_wrong_answer': 'compile이 학습 실행이라고 쓰거나, optimizer가 gradient를 계산한다고 쓴다.', 'likely_misconception': 'Keras API 단계를 직접 구현 학습 루프의 책임과 대응시키지 못해서다.', 'correction_hint_only': 'Keras 연결 문항의 핵심은 문법 암기가 아니라 직접 구현 구조와 책임 단위의 대응입니다.', 'retry_prompt': '3-class toy data라면 output layer, loss, y 표현이 어떻게 바뀌는지 쓰시오.', 'gate_node': 'G2-P07, 연결 n_ML13.keras', 'source_lecture': '13강 연결'}]
feedback_df = pd.DataFrame(feedback_rows)
required_columns = [
    "question_id",
    "assessed_concept",
    "expected_answer_type",
    "common_wrong_answer",
    "likely_misconception",
    "correction_hint_only",
    "retry_prompt",
    "gate_node",
    "source_lecture",
]
assert list(feedback_df.columns) == required_columns
feedback_df


| question_id | assessed_concept | expected_answer_type | common_wrong_answer | likely_misconception | correction_hint_only | retry_prompt | gate_node | source_lecture |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Q01-A | G1-P02 | 개념+수식+shape 설명 | activation을 threshold로만 설명하거나, MLP를 단순히 Perceptron을 많이 더한 것으로만 설명한다. | weighted sum과 nonlinear transformation의 역할을 분리하지 못해서 생기는 오답이다. | 계산식에는 `w, x, b`가, 구조 설명에는 hidden layer와 nonlinear activation이 반드시 들어가야 합니다. | `x`가 3차원이고 hidden neuron이 4개라면 `W`와 `b` shape를 쓰시오. | G1-P02 | 8강 |
| Q02-B | G1-P02 | 개념+수식+shape 설명 | “XOR가 비선형이라서”만 쓰고 왜 선형 분리 불가능한지 설명하지 않는다. | 데이터 배치와 decision boundary를 연결하지 못했기 때문이다. | XOR 설명에는 “대각선 배치”와 “hidden representation 변환”이 함께 있어야 합니다. | NAND와 OR 출력을 hidden feature로 만들면 XOR output을 어떤 논리로 얻는지 쓰시오. | G1-P02 | 8강 |
| Q03-C | G1-P03, G1-P05, 연결 n_ML13.keras | 개념+수식+shape 설명 | optimizer가 gradient를 계산한다고 쓰거나, gradient를 감소 방향으로 착각한다. | backprop과 optimizer step을 한 덩어리로 암기했기 때문이다. | gradient 방향과 update 방향을 반드시 반대로 구분하세요. | `W <- W + lr*dW`가 왜 보통 틀린 update인지 설명하시오. | G1-P03, G1-P05, 연결 n_ML13.keras | 10강 |
| Q04-D | G1-P04 | 개념+수식+shape 설명 | local derivative만 쓰고 upstream gradient를 빠뜨린다. | 미분 공식을 함수 하나의 기울기로만 보고 계산 그래프의 연결을 보지 못해서다. | backprop 문장에는 upstream gradient와 local derivative가 모두 필요합니다. | `L -> a -> z -> W` 경로에서 각 local derivative 이름을 쓰시오. | G1-P04 | 10강 |
| Q05-E | G1-P05 | 개념+수식+shape 설명 | cache와 gradient를 같은 것으로 쓰거나, update에서 gradient를 새로 계산한다고 쓴다. | 학습 루프의 책임 분리가 흐려졌기 때문이다. | cache는 backward 계산을 위해 forward 중간값을 보관하는 장치입니다. | `a1`을 cache하지 않으면 출력층 `dW2`와 sigmoid derivative 계산에서 무엇이 불편한지 쓰시오. | G1-P05 | 11강 |
| Q06-F0 | G2-P01 | 개념+수식+shape 설명 | `W1`을 `(input, hidden)`으로 두고 `x @ W1` convention과 섞는다. | 단일 샘플 convention과 batch convention을 동시에 섞었기 때문이다. | 먼저 convention을 고정한 뒤 모든 shape를 거기에 맞춰야 합니다. | hidden neuron이 3개라면 `W1`, `b1`, `W2` shape가 어떻게 바뀌는가? | G2-P01 | 11강 |
| Q07-F | G2-P01 | 개념+수식+shape 설명 | 출력층에도 sigmoid를 적용하거나 loss에서 1/2를 빼먹는다. | 문제의 output layer가 linear라는 조건을 놓쳤기 때문이다. | forward cache 값은 이후 backward에서 그대로 재사용됩니다. | `y=1`이라면 같은 `a2`에서 loss만 다시 계산하시오. | G2-P01 | 11강 |
| Q08-Bridge | G1-P04, G2-P02~P03 | 개념+수식+shape 설명 | 은닉층 gradient를 출력층처럼 `delta2*x`로 계산한다. | 은닉층이 loss에 직접 닿지 않고 출력층을 통해 간접 연결된다는 점을 놓친다. | 은닉층에서는 반드시 출력층 weight와 activation local derivative가 끼어듭니다. | `W1[1,0]`에서 loss까지 가는 경로를 쓰시오. | G1-P04, G2-P02~P03 | 11강 |
| Q09-G | G2-P02 | 개념+수식+shape 설명 | `delta2 = y-a2`로 부호를 반대로 쓴다. | 내부 미분 `d(y-a2)/da2 = -1`을 놓쳐서 생긴다. | 출력층 delta의 부호는 update 방향 전체를 바꾸므로 반드시 확인해야 합니다. | `a2=0.2, y=1`이면 `delta2`의 부호는 무엇인가? | G2-P02 | 11강 |
| Q10-H | G2-P03 | 개념+수식+shape 설명 | `W2 @ delta2`를 쓰거나 sigmoid derivative 위치에 `@`를 쓴다. | forward 방향과 backward 방향이 반대라는 점과 elementwise derivative를 구분하지 못해서다. | 은닉 delta의 핵심은 `W2.T`와 sigmoid local derivative를 분리해서 쓰는 것입니다. | `W2`가 `(1,3)`이면 `W2.T @ delta2` 결과 shape는 무엇인가? | G2-P03 | 11강 |
| Q11-I | G2-P04 전 단계 | 개념+수식+shape 설명 | `dW1`을 `delta1 * x`로 써서 `(2,)` 벡터처럼 처리한다. | 단일 예제에서 x 값이 모두 1이라 행렬 구조가 눈에 덜 보인다. | 수치가 우연히 같아져도 gradient tensor의 shape는 보존해야 합니다. | `x=[2,-1]`이면 `dW1`의 각 row가 어떻게 달라지는지 식으로 쓰시오. | G2-P04 전 단계 | 11강 |
| Q12-BackpropSummary | G2-P02~P04 | 개념+수식+shape 설명 | gradient check를 validation accuracy나 loss 감소 확인으로 설명한다. | 구현 검증과 모델 성능 평가를 혼동했기 때문이다. | gradient check는 학습 결과가 아니라 backward 구현의 신뢰도를 검증합니다. | gradient check가 실패했을 때 `W2.T`, 부호, `*`/`@` 중 무엇을 먼저 볼지 순서를 정하시오. | G2-P02~P04 | 11강 |
| Q13-JK | G2-P04, G2-P05 | 개념+수식+shape 설명 | `dW = X.T @ dZ`를 쓰고 shape `(D,H)`로 만든다. | W convention을 `(D,H)`로 둔 다른 교재 식과 섞었기 때문이다. | Dense backward 공식은 convention에 따라 달라 보이므로 shape 검산이 필수입니다. | `B=5,D=4,H=3`일 때 `dZ.T @ X` shape를 쓰시오. | G2-P04, G2-P05 | 12강 |
| Q14-L | G2-P05 | 개념+수식+shape 설명 | softmax를 class별 독립 sigmoid로 생각하거나 `/B` 위치를 loss 평균과 분리하지 못한다. | 확률분포의 class 축 합 1 조건과 batch 평균 convention을 놓쳤기 때문이다. | SoftmaxCE backward의 간단한 형태는 softmax와 CE를 결합했을 때 나옵니다. | `B=4,K=3`이면 logits와 one-hot target shape를 쓰시오. | G2-P05 | 12강 |
| Q15-MN | G2-P06, G2-P07 | 개념+수식+shape 설명 | batch 축을 feature 축으로 보고 `axis=1`로 bias gradient를 더한다. | batch dimension 0번째 축이라는 convention을 고정하지 않았기 때문이다. | Network abstraction은 chain rule을 layer list 순회 구조로 고정한 것입니다. | `B=8,D=2,H=4`에서 `X`, `W`, `Z`, `dW` shape를 쓰시오. | G2-P06, G2-P07 | 12강 |
| Q16-OP | G2-P07, 연결 n_ML13.keras | 종합 설계 판단 | compile이 학습 실행이라고 쓰거나, optimizer가 gradient를 계산한다고 쓴다. | Keras API 단계를 직접 구현 학습 루프의 책임과 대응시키지 못해서다. | Keras 연결 문항의 핵심은 문법 암기가 아니라 직접 구현 구조와 책임 단위의 대응입니다. | 3-class toy data라면 output layer, loss, y 표현이 어떻게 바뀌는지 쓰시오. | G2-P07, 연결 n_ML13.keras | 13강 연결 |


## 재시도 미니문제 모음

1. Perceptron 3-input, hidden 4 구조에서 `W`, `b`, `z` shape를 쓰시오.
2. XOR 좌표를 그리고 직선 하나로 나눌 수 없는 이유를 3문장으로 쓰시오.
3. `L=0.5(y-a)^2`에서 `dL/da`를 부호까지 유도하시오.
4. `W2`가 `(1,3)`일 때 hidden delta shape를 쓰시오.
5. `B=8,D=2,H=4`에서 `dW=dZ.T@X`의 shape를 검산하시오.
6. Binary classification과 3-class classification의 output/loss/metric 차이를 쓰시오.


## 최종 Gate 통과 체크

- [ ] `z = w^T x + b`, activation, MLP, XOR 한계를 한 흐름으로 설명할 수 있다.
- [ ] gradient는 증가 방향이고 gradient descent는 반대 방향임을 설명할 수 있다.
- [ ] `delta2 = a2-y`, `dL/da1 = W2.T @ delta2`, `delta1 = dL/da1 * a1*(1-a1)`를 부호와 shape까지 쓸 수 있다.
- [ ] `Dense.backward`에서 `dW`, `db`, `dX`의 목적과 shape를 구분할 수 있다.
- [ ] activation layer는 parameter update 대상이 아님을 설명할 수 있다.
- [ ] mini-batch의 0번째 축과 `dZ.T @ X`를 연결할 수 있다.
- [ ] `Network.forward` 순방향, `Network.backward` 역방향을 chain rule 구조로 설명할 수 있다.
- [ ] `Network()`와 Keras `Sequential()`, 직접 train loop와 `model.fit()`을 대응시킬 수 있다.
- [ ] Optimizer는 gradient를 계산하는 존재가 아니라, 계산된 gradient로 parameter를 update하는 규칙이라고 말할 수 있다.
